In [37]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/nlp-getting-started/sample_submission.csv
/kaggle/input/competitions/nlp-getting-started/train.csv
/kaggle/input/competitions/nlp-getting-started/test.csv


In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
import zipfile
from catboost import CatBoostClassifier
from google.colab import files
import lightgbm as lgb
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import tensorflow as tf
from keras.models import Sequential
from tensorflow.keras import layers
from sklearn.model_selection import RandomizedSearchCV
# from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Activation, BatchNormalization
# from keras.layers import Input
# # السطر الجديد اللي هيحل الأزمة:
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import re
# الرقم 3 ده بيقول للبرنامج: "لو سمحت، متطلعليش أي رسائل تحذيرية، طلعلي بس لو فيه مصيبة (Error) حقيقية".

In [39]:
# جميع الملفات الموجودة في /kaggle/input
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/nlp-getting-started/sample_submission.csv
/kaggle/input/competitions/nlp-getting-started/train.csv
/kaggle/input/competitions/nlp-getting-started/test.csv


In [40]:
train=pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
test=pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

In [41]:
print(train.head())
print(train.shape)
print(train.columns)
print(train.describe())
print(train.info())
train.isna().sum()
# المصطلح non-nullده يعني إن الخانة "ليست فارغة"؛ أي أنها تحتوي على قيمة حقيقية (رقم، نص، أو تاريخ) وليست NaN

   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  
(7613, 5)
Index(['id', 'keyword', 'location', 'text', 'target'], dtype='object')
                 id      target
count   7613.000000  7613.00000
mean    5441.934848     0.42966
std     3137.116090     0.49506
min        1.000000     0.00000
25%     2734.000000     0.00000
50%     5408.000000     0.00000
75%     8146.000000     1.00000
max    10873.000000     1.00000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [42]:
print(test.head())
print(test.shape)
print(test.columns)
print(test.describe())
print(test.info())
test.isna().sum()

   id keyword location                                               text
0   0     NaN      NaN                 Just happened a terrible car crash
1   2     NaN      NaN  Heard about #earthquake is different cities, s...
2   3     NaN      NaN  there is a forest fire at spot pond, geese are...
3   9     NaN      NaN           Apocalypse lighting. #Spokane #wildfires
4  11     NaN      NaN      Typhoon Soudelor kills 28 in China and Taiwan
(3263, 4)
Index(['id', 'keyword', 'location', 'text'], dtype='object')
                 id
count   3263.000000
mean    5427.152927
std     3146.427221
min        0.000000
25%     2683.000000
50%     5500.000000
75%     8176.000000
max    10875.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3263 entries, 0 to 3262
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        3263 non-null   int64 
 1   keyword   3237 non-null   object
 2   location  2158 non-null   object
 3   text

id             0
keyword       26
location    1105
text           0
dtype: int64

In [43]:
Y_train=train.target
Y_train = (Y_train >= 0.5).astype(int)
X_train=train.drop(columns=["target"],axis=1)
X_test=test
# الtarget مينفعش نحطه في الpipeline لانه اصلا متظبط هنا والقيم كلها موجوده و0او 1

In [44]:
import re
# 1. عد علامات التعجب (مؤشر للذعر)
train['exclamation_count'] = train['text'].apply(lambda x: x.count('!'))

# 2. عد الكلمات اللي كلها حروف كبيرة (مؤشر للصراخ الرقمي)
# بنقسم النص لكلمات ونعد اللي حروفها كلها Upper
train['caps_count'] = train['text'].apply(lambda x: len([w for w in x.split() if w.isupper() and len(w) > 1]))

# 3. طول التغريدة (هل البلاغات الحقيقية أطول ولا أقصر؟)
train['text_length'] = train['text'].apply(len)

def clean_special_chars(text):
    # الـ Regex ده بيقول: أي حاجة مش حرف (a-z) ومش رقم (0-9) حط مكانها مسافة
    # [^a-zA-Z0-9] تعني: "أي شيء ليس حرفاً أو رقماً"
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # خطوة إضافية: لو فيه مسافات كتير ورا بعض بيخليها مسافة واحدة بس
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

# لتطبيق الكود على عمود النص في الـ DataFrame بتاعك:
train['text'] = train['text'].apply(clean_special_chars)

def advanced_cleaning(text):
    # 1. تنظيف الحروف المتكررة (أكثر من مرتين)
    # الـ Regex ده بيدور على أي حرف ( \w ) متكرر ( \1 ) أكتر من مرتين {2,}
    cleaning_text = re.sub(r'(\w)\1{2,}', r'\1\1', text)
    
    # 2. توحيد حالة الحروف لصغيرة عشان المقارنة
    cleaning_text = cleaning_text.lower()
    
    return cleaning_text
# اتأكدي دايماً إنك في الـ apply بتنادي على الدالة دي، وبتحطي الناتج في عمود جديد (زي text_cleaned) مش بتمسحي العمود الأصلي، عشان لو احتاجتي ترجعي للأصل في أي وقت.
# تطبيق التنظيف الأساسي أولاً
train['text_cleaned'] = train['text'].apply(advanced_cleaning)

# 3. خلق الـ Features الجديدة (الأفكار بتاعتك)
# كلمات "المزاح" أو غير الرسمية
fake_keywords = ['zombie', 'girlfriend', 'baby', 'haha', 'loool', 'bro', 'buddy', 'game', 'song', 'cancelled',"film","bag","cover","wow","alright"
                ,'album', 'lyrics', 'watch', 'video', 'check out my photo']
train['has_fake_keyword'] = train['text_cleaned'].apply(lambda x: 1 if any(word in x for word in fake_keywords) else 0)

# كلمات "الجدية" والبلاغات
real_keywords = ['legit', 'real', 'warning', 'radar', 'breaking', 'special weather statement',"pkk suicide bomber","storm prediction center","nasa"
                 'burn', 'fire', 'die', 'kill', 'flood', 'crash', 'collapse', 
                   'destroy', 'evacuate', 'rescue', 'explode', 'panic', 'shaking',"Urgent","ntsb"]
train['has_real_keyword'] = train['text_cleaned'].apply(lambda x: 1 if any(word in x for word in real_keywords) else 0)

# البحث عن الوقت (مثلاً 10:30 am أو 22:00)
train['mentions_time'] = train['text'].apply(lambda x: 1 if re.search(r'\d{1,2}:\d{2}', x) else 0)

In [45]:
# 1. عد علامات التعجب (مؤشر للذعر)
test['exclamation_count'] = test['text'].apply(lambda x: x.count('!'))

# 2. عد الكلمات اللي كلها حروف كبيرة (مؤشر للصراخ الرقمي)
# بنقسم النص لكلمات ونعد اللي حروفها كلها Upper
test['caps_count'] = test['text'].apply(lambda x: len([w for w in x.split() if w.isupper() and len(w) > 1]))

# 3. طول التغريدة (هل البلاغات الحقيقية أطول ولا أقصر؟)
test['text_length'] = test['text'].apply(len)

def clean_special_chars(text):
    # الـ Regex ده بيقول: أي حاجة مش حرف (a-z) ومش رقم (0-9) حط مكانها مسافة
    # [^a-zA-Z0-9] تعني: "أي شيء ليس حرفاً أو رقماً"
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # خطوة إضافية: لو فيه مسافات كتير ورا بعض بيخليها مسافة واحدة بس
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

# لتطبيق الكود على عمود النص في الـ DataFrame بتاعك:
test['text'] = test['text'].apply(clean_special_chars)

def advanced_cleaning(text):
    # 1. تنظيف الحروف المتكررة (أكثر من مرتين)
    # الـ Regex ده بيدور على أي حرف ( \w ) متكرر ( \1 ) أكتر من مرتين {2,}
    cleaning_text = re.sub(r'(\w)\1{2,}', r'\1\1', text)
    
    # 2. توحيد حالة الحروف لصغيرة عشان المقارنة
    cleaning_text = cleaning_text.lower()
    
    return cleaning_text
# اتأكدي دايماً إنك في الـ apply بتنادي على الدالة دي، وبتحطي الناتج في عمود جديد (زي text_cleaned) مش بتمسحي العمود الأصلي، عشان لو احتاجتي ترجعي للأصل في أي وقت.
# تطبيق التنظيف الأساسي أولاً
test['text_cleaned'] = test['text'].apply(advanced_cleaning)

# 3. خلق الـ Features الجديدة (الأفكار بتاعتك)
# كلمات "المزاح" أو غير الرسمية
fake_keywords = ['zombie', 'girlfriend', 'baby', 'haha', 'loool', 'bro', 'buddy', 'game', 'song', 'cancelled',"film","bag","cover","wow","alright"
                ,'album', 'lyrics', 'watch', 'video', 'check out my photo']
test['has_fake_keyword'] = test['text_cleaned'].apply(lambda x: 1 if any(word in x for word in fake_keywords) else 0)

# كلمات "الجدية" والبلاغات
real_keywords = ['legit', 'real', 'warning', 'radar', 'breaking', 'special weather statement',"pkk suicide bomber","storm prediction center","nasa"
                 'burn', 'fire', 'die', 'kill', 'flood', 'crash', 'collapse', 
                   'destroy', 'evacuate', 'rescue', 'explode', 'panic', 'shaking',"run","collapse","burn","help","Urgent","ntsb"]
test['has_real_keyword'] = test['text_cleaned'].apply(lambda x: 1 if any(word in x for word in real_keywords) else 0)

# البحث عن الوقت (مثلاً 10:30 am أو 22:00)
test['mentions_time'] = test['text'].apply(lambda x: 1 if re.search(r'\d{1,2}:\d{2}', x) else 0)

In [46]:
import nltk
from nltk.corpus import stopwords

# متخزنة في المكتبة بحروف صغيرة (lowercase). تحميل الكلمات الشائعة لأول مرة
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    # بنقسم النص لكلمات، وبناخد الكلمة بس لو مش موجودة في لستة الـ stopwords
    words = text.split()
    cleaned_words = [w for w in words if w.lower() not in stop_words]
    return " ".join(cleaned_words)

# تطبيق الكود على العمود المنظف بتاعك
train['text_cleaned'] = train['text_cleaned'].apply(remove_stopwords)
# بنحسب عدد الكلمات في كل سطر
# وبنحتفظ بس بالسطور اللي عدد كلماتها أكبر من 2
train = train[train['text_cleaned'].apply(lambda x: len(x.split()) > 2)]

# ملاحظة: الخطوة دي بتتعمل في الـ Train بس عشان نعلم الموديل على بيانات نظيفة
# لكن في الـ Test بنسيب كل السطور عشان نطلع نتايج لكل الـ IDs المطلوبة

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [47]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    # بنقسم النص لكلمات، وبناخد الكلمة بس لو مش موجودة في لستة الـ stopwords
    words = text.split()
    cleaned_words = [w for w in words if w.lower() not in stop_words]
    return " ".join(cleaned_words)
# تطبيق الكود على العمود المنظف بتاعك
test['text_cleaned'] = test['text_cleaned'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [48]:
import re
def replace_urls_with_placeholder(text):
    # التعديل هنا: غيرنا '' بـ ' URL_LINK '
    # الـ Regex هيبدل أي رابط بالكلمة دي
    cleaned_text = re.sub(r'http\S+|www\S+|https\S+', ' URL_LINK ', text, flags=re.MULTILINE)
    
    # خطوة إضافية: تنظيف المسافات الزايدة اللي ممكن تطلع
    cleaned_text = ' '.join(cleaned_text.split())
    
    return cleaned_text

# تطبيق الكود المعدل
train['text_cleaned'] = train['text_cleaned'].apply(replace_urls_with_placeholder)
test['text_cleaned'] = test['text_cleaned'].apply(replace_urls_with_placeholder)

from textblob import TextBlob

# 1. التنوع اللفظي (Lexical Diversity)
def get_lexical_diversity(text):
    words = text.split()
    if len(words) == 0: return 0
    return len(set(words)) / len(words)

# 2. تحليل المشاعر (Sentiment Analysis)
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

#     reg_alpha (الفرامل الأولى - L1):
# دي بنسميها "المقص". بتعمل حاجة ذكية جداً؛ لو لقت Feature (عمود) في الداتا ملوش لازمة أو تأثيره ضعيف، بتقلل قيمته لحد ما تخليه صفر. كأنها بتقول للموديل: "ركز في المهم وسيبك من الرغي".
# reg_lambda (الفرامل الثانية - L2):
# دي "المهدئ". بدل ما تشيل الـ Feature خالص، بتخلي الأوزان (Weights) بتاعة الموديل كلها صغيرة ومتوازنة. بتمنع إن الموديل "يتجنن" ويركز على كلمة واحدة بس ويفتكر إنها هي السر
# عدد الكلمات الفريدة ÷ عدد الكلمات الكلي.مثال: لو واحد كتب "Fire Fire Fire" (تكرار)، الـ Diversity هتكون $1/3 = 0.33$. لكن لو كتب "Big Fire burning" (تنوع)، هتكون $3/3 = 1.0$.الهدف: كشف الـ Spam أو الهزار اللي بيعتمد على تكرار الكلمات.
train['lexical_diversity'] = train['text_cleaned'].apply(get_lexical_diversity)
train['sentiment'] = train['text'].apply(get_sentiment)

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

# التعديل هنا في الـ ngram_range
tfidf = TfidfVectorizer(
    max_features=5000, 
    stop_words='english', 
    ngram_range=(1, 3) # السحر هنا! 1 يعني كلمة واحدة، 2 يعني كلمتين ورا بعض
)

# وبعدين بتكملي عادي
X_train_tfidf = tfidf.fit_transform(train['text_cleaned'])
X_test_tfidf = tfidf.transform(test['text_cleaned'])

emergency_verbs = ['burn', 'fire', 'die', 'kill', 'flood', 'crash', 'collapse', 
                   'destroy', 'evacuate', 'rescue', 'explode', 'panic', 'shaking']
def count_emergency_verbs(text):
    text = text.lower()
    return sum(1 for verb in emergency_verbs if verb in text)

train['emergency_verb_count'] = train['text_cleaned'].apply(count_emergency_verbs)
test['emergency_verb_count'] = test['text_cleaned'].apply(count_emergency_verbs)

In [50]:
numeric_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())])
cat_transformer=Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
#     تخيلي إن الموديل اتعلم على مدن (القاهرة، الإسكندرية، بنها). وفجأة وأنتي بتعملي Test، ظهرت مدينة "المنصورة" والموديل مش عارفها!
# الحل: السطر ده بيقول للموديل: "لو شفت قيمة جديدة معرفتهاش وأحنا بنتدرب، متضربش Error.. اعتبرها قيمة مجهولة واديها رقم محدد (زي -1 أو 0) وكمل شغلك عادي". ده بيخلي الكود بتاعك Robust (صلب) ومبيقع
    ("encoder", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
# 1. الأرقام (الـ id بس، وممكن نشيله خالص لأنه ملوش لازمة في التدريب)
numeric_columns = ["id"] 

# 2. الكلمات التصنيفية (keyword, location)
categorical_columns = ["location", "keyword"]

# 3. النص (العمود الأهم)
text_column = "text"

# الـ Pipeline بتاع النص (بيحول الجمل لأرقام ذكية)
from sklearn.feature_extraction.text import TfidfVectorizer
text_transformer = TfidfVectorizer(max_features=5000) # بياخد أهم 5000 كلمة

preprocessing = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", cat_transformer, categorical_columns),
        ("text", text_transformer, text_column) # إضافة معالج النصوص
    ])

In [51]:
models_Classifier={
    "RF": RandomForestClassifier(),
    "XGB": XGBClassifier(tree_method='hist',verbosity=0),
    "LGBM": LGBMClassifier()
    }
# إمتى تقفلي الـ GPU (وتستخدمي الـ CPU)؟
# حجم الداتا صغير: لو عدد الصفوف عندك أقل من 50 ألف صف، الـ CPU هيكون أسرع بكتير. ليه؟ لأن نقل البيانات من الرام للـ GPU بياخد وقت أطول من عملية الحساب نفسها في الداتا الصغيرة.
# كثرة التحذيرات والـ Overhead: لو الجهاز بيقعد يعمل "Initialize" للـ GPU مع كل لفة في الـ GridSearch، ده بيعطل الشغل جداً.
# 2. إمتى تسيبي الـ GPU شغال؟
# الداتا ضخمة: (مئات الآلاف أو الملايين من الصفوف).
# الـ n_estimators كتير: لو بتجربي 2000 شجرة أو أكتر، هنا الـ GPU هيبدأ يظهر قوته.
param_grid={
    "RF": {
        "model__n_estimators": [200,250,500],
        "model__max_depth": [10,5,8]},
    "XGB": {
        "model__n_estimators": [400,300,500],#كلما نزود الاشجار يبقى افضل ونقلل العمق
        "model__learning_rate": [0.02, 0.001],
        "model__max_depth": [4,5,8,9],
        "model__reg_alpha": [0, 0.1, 0.05],  # الفرامل الأولى (L1)
        "model__reg_lambda": [1, 1.5, 2],   # الفرامل الثانية (L2)
        "model__tree_method":['hist']},
# بدل ما الموديل يقعد يحسب كل نقطة داتا لوحدها (وده بياخد وقت ومجهود كبير من الـ CPU)، بيقسم الأرقام لمجموعات أو "صناديق" (Bins).
    "LGBM":{
        "model__n_estimators": [200,250,500],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [4,10,5,8],
        "model__num_leaves": [15,31,63],
        "model__subsample": [0.7,0.8,1],
        "model__verbose":[-1]}
        }
best_models={}

In [52]:
for name,model in models_Classifier.items():
  pipeline=Pipeline(steps=[
      ("preprocessing",preprocessing),
      ("model",model)])
# مينفعش نعمل GPUعلى البيانات الصغيرة لانها بتخليها ابطا 
# دايما استخدمي RandomizedCV لانه اسرع وافضل وحتى لو استخدمتيGPU مش هيبقى حلو زيه مع الML
  random_search=RandomizedSearchCV(
      estimator=pipeline,cv=3,scoring="accuracy",param_distributions=param_grid[name],n_iter=2,n_jobs=-1) 
  random_search.fit(X_train,Y_train)
  print(name,random_search.best_score_)
  best_models[name] = random_search.best_estimator_
# from sklearn.ensemble import VotingClassifier

# # هناخد أحسن نسخ طلعوا من الـ RandomizedSearch
# voting_model = VotingClassifier(
#     estimators=[
#         ('rf', best_models['RF']),
#         ('xgb', best_models['XGB']),
#         ('lgbm', best_models['LGBM'])
#     ],
#     voting='soft' # 'soft' بيحسب الاحتمالات بدقة أكبر
# )

# voting_model.fit(X_train, Y_train)

RF 0.6209210073885222
XGB 0.5738866716385257


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

LGBM 0.5447179484630879


In [53]:
bs_model=best_models["XGB"]
lgb_model=best_models["RF"]
bs_model_steps = bs_model.named_steps["model"]
importances = bs_model_steps.feature_importances_
for col, imp in zip(X_train.columns, importances):
    print(col, imp)
lgb_model_steps = lgb_model.named_steps["model"]
importances = lgb_model_steps.feature_importances_
for col, imp in zip(X_train.columns, importances):
    print(col, imp)
pred1=bs_model.predict(X_test)
pred2=lgb_model.predict(X_test)
final=(pred1 + pred2)/2
print(f"Current predictions count: {len(final)}")

id 0.006020653
keyword 0.0016496679
location 0.018437868
text 0.0
id 0.02203875542216473
keyword 0.0034178376203751444
location 0.016849777732556317
text 0.0005821382344613371
Current predictions count: 3263


In [54]:
final = (final >= 0.5).astype(int)
submission = pd.DataFrame({
    "id": X_test["id"],
    "target": final })
submission.to_csv("submission.csv", index=False)
print("Submission file created!")
print(submission.head())

Submission file created!
   id  target
0   0       0
1   2       0
2   3       1
3   9       0
4  11       0
